# Sanad سند — question answering over Saudi law

## Part 2: fine-tuning the retriever

Part 1 showed that BM25 wins on LLM-written questions, which copy the wording of the law, while embedding models win on the 79 questions real people asked. Here a multilingual embedding model (multilingual-e5-base) is fine-tuned two ways from the same starting point:

- **questions**: ArabLegalEval training questions paired with the passage each was written from
- **questions+cases**: the same, plus ALARB court cases, where the facts of a case are paired with an article the court cited

No training pair touches a passage that answers a dev, test or Najiz question. The model kept is the one with the best dev MRR@10; test and Najiz are scored once.

Code: https://github.com/readaloush/sanad

**Settings:** Internet on, GPU accelerator.

In [ ]:
!pip -q install sentence-transformers rank_bm25
!git clone -q https://github.com/readaloush/sanad.git
%cd sanad
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Train and evaluate

In [ ]:
!python -m sanad.finetune_retriever --base intfloat/multilingual-e5-base --epochs 2 --data data --out results 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$|Loading weights'

## Keep the results and the selected model

In [ ]:
!mkdir -p /kaggle/working/out && cp results/retriever.json /kaggle/working/out/ && cp -r results/retriever /kaggle/working/out/ 2>/dev/null; true
%cd /kaggle/working
!rm -rf /kaggle/working/sanad && ls -la /kaggle/working/out /kaggle/working/out/retriever 2>/dev/null